# Protocolo de 5 tareas en GPU (Kaggle)

Ejecuta `planes/plan_tareas.json` (5 tareas de 20 clases, 5 configuraciones de replay, buffers 0/10/20 %;
ver `docs/plan_cinco_tareas.md`) con el orquestador del proyecto, **una semilla detrás de otra**:

1. **`planes/plan_prueba_tareas.json`**: el plan en miniatura (`--quick`), unos minutos. Si algo falla en GPU, sale aquí.
2. **`plan_tareas` semilla a semilla** (`SEMILLAS`): cada semilla completa sus 5 trabajos antes de empezar la siguiente.
   Al pasar `HORAS` desde el inicio no se empiezan tareas nuevas; las que estén en curso terminan.
   Estimación sin medir: unas 4 h por semilla en 2 × T4.

Al final se muestra el informe (`INFORME.md`) y todo queda en `resultados_kaggle_tareas.tar.gz` (pestaña *Output*).
Las semillas que falten se lanzan en otra sesión cambiando `SEMILLAS`; los paquetes se combinan descomprimiéndolos
en la misma carpeta y regenerando el informe con `python informe_tareas.py planes/plan_tareas.json`.

**Antes de ejecutar** (panel *Session options*): **Accelerator → GPU T4 x2** e **Internet → On**.
Para que siga aunque cierres el navegador: *Save Version* → *Save & Run All (Commit)*. Límite de Kaggle: 12 h.

In [ ]:
# ---------------------------------------------------------------- parámetros
import time
T0 = time.time()
SEMILLAS = [42, 43]   # en este orden; la segunda solo empieza si queda tiempo
HORAS = 9.5           # desde el inicio del cuaderno: después no se empiezan tareas nuevas (Kaggle corta a las 12 h)
PARALELO = 4          # tareas a la vez: 2 por GPU
PRUEBA = True         # plan en miniatura antes del real
COMMIT = "6243ae6"    # revisión del código en GitHub (debe estar subida)

In [ ]:
!nvidia-smi -L

In [ ]:
%%bash -s "$COMMIT"
set -e
cd /kaggle/working
rm -rf cfl-practicas
git clone -q https://github.com/Esteban7754/cfl-practicas.git
cd cfl-practicas && git checkout -q "$1" && git log -1 --oneline
test -f cfl_tareas.py && test -f planes/plan_tareas.json || { echo "[ERROR] El commit no tiene el protocolo de 5 tareas"; exit 1; }

## Entorno: mismas versiones fijadas que la imagen Docker, con PyTorch CUDA

In [ ]:
%%bash
set -e
cd /kaggle/working/cfl-practicas
pip install -q uv
uv venv -q -p 3.13 /tmp/venv
sed -e '/^--extra-index-url/d' -e 's/+cpu//' requirements.lock > /tmp/req-cuda.txt
uv pip install -q -p /tmp/venv/bin/python -r /tmp/req-cuda.txt

ok() { /tmp/venv/bin/python -c "import torch; x = torch.ones(8, device='cuda'); assert float((x * 2).sum()) == 16" 2>/dev/null; }
if ! ok; then
    TORCH=$(grep '^torch==' /tmp/req-cuda.txt); VISION=$(grep '^torchvision==' /tmp/req-cuda.txt)
    for cu in cu130 cu128 cu126; do
        echo "PyTorch de PyPI no usa la GPU; probando $TORCH con $cu"
        uv pip install -q -p /tmp/venv/bin/python --reinstall "$TORCH" "$VISION" \
            --index-url "https://download.pytorch.org/whl/$cu" && ok && break || true
    done
fi
ok || { echo "[ERROR] PyTorch no puede usar la GPU. ¿Accelerator = GPU T4 x2?"; exit 1; }
/tmp/venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

In [ ]:
import os, subprocess
os.environ.pop("PYTHONPATH", None)   # sin el sitecustomize de Kaggle
os.environ.update(HF_HOME="/tmp/hf", MPLBACKEND="Agg", PYTHONHASHSEED="0", PYTHONUNBUFFERED="1",
                  CUBLAS_WORKSPACE_CONFIG=":4096:8")
REPO = "/kaggle/working/cfl-practicas"
PY = "/tmp/venv/bin/python"

# CIFAR-100 en la revisión fijada; después todo trabaja sin red.
!cd {REPO} && {PY} docker/descargar_datos.py
os.environ["CFL_DATA_PRELOADED"] = "1"
!cd {REPO} && {PY} cfl.py info

In [ ]:
# Los tests ejecutan el pipeline en GPU: si algo falla en CUDA se ve aquí y no se lanza nada.
r = subprocess.run([PY, "-m", "unittest"], cwd=REPO, capture_output=True, text=True)
lines = [l for l in r.stderr.splitlines() if "sitecustomize" not in l and "wrapt" not in l]
print("\n".join(lines[-40:] if r.returncode else lines[-4:]))
assert r.returncode == 0, "Los tests fallan en este entorno: no se lanza el plan"
print(f"Preparación: {(time.time() - T0) / 60:.0f} min")

## Planes

El orquestador informa cada 10 minutos de las tareas en curso y su ronda. Todas las semillas escriben en
`resultados/plan_tareas/` (una carpeta `seed_<n>` por trabajo y semilla) y el informe se regenera con todo lo que haya.
Código de salida 3 = pausa por el límite de horas (lo terminado queda guardado).

In [ ]:
import json
from pathlib import Path

def ejecutar(plan, *extra):
    print(f"\n################ {plan} {' '.join(extra)} ################", flush=True)
    proc = subprocess.Popen([PY, "plan_experimentos.py", plan, "--informe-cada", "10", "--paralelo", str(PARALELO), *extra],
                            cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if "sitecustomize" not in line and "wrapt" not in line:
            print(line, end="", flush=True)
    return proc.wait()

codigos = {}
if PRUEBA:
    codigos["prueba"] = ejecutar("planes/plan_prueba_tareas.json")
    assert codigos["prueba"] == 0, "El plan en miniatura falla: no se lanza el real"

base = json.loads((Path(REPO) / "planes/plan_tareas.json").read_text(encoding="utf-8"))
for seed in SEMILLAS:
    quedan = HORAS - (time.time() - T0) / 3600
    if quedan < 0.5:
        print(f"\nSin tiempo para la semilla {seed} (quedan {quedan:.1f} h): se lanzará en otra sesión.")
        break
    plan = {**base, "trabajos": [{**job, "semillas": [seed]} for job in base["trabajos"]]}
    path = f"planes/_kaggle_seed{seed}.json"
    (Path(REPO) / path).write_text(json.dumps(plan, indent=1, ensure_ascii=False), encoding="utf-8")
    codigos[seed] = ejecutar(path, "--horas", f"{quedan:.2f}")
    if codigos[seed] == 3:
        print(f"\nPausa por el límite de horas durante la semilla {seed}.")
        break
print("\nCódigos de salida:", codigos, "| tiempo total:", f"{(time.time() - T0) / 3600:.2f} h")

In [ ]:
# Informe con todas las semillas terminadas y paquete para descargar (sin los pesos .pt, que ocupan mucho).
subprocess.run([PY, "informe_tareas.py", "planes/plan_tareas.json"], cwd=REPO, check=True)
informe = Path(REPO) / "resultados/plan_tareas/INFORME.md"
print(informe.read_text() if informe.is_file() else "(sin informe)")
carpetas = [c for c in ("resultados/plan_tareas", "resultados/plan_prueba_tareas") if (Path(REPO) / c).is_dir()]
subprocess.run(["tar", "-czf", "/kaggle/working/resultados_kaggle_tareas.tar.gz", "--exclude=*.pt", "-C", REPO, *carpetas],
               check=True)
print("\nPaquete: /kaggle/working/resultados_kaggle_tareas.tar.gz")
fallos = {k: v for k, v in codigos.items() if v not in (0, 3)}
assert not fallos, f"Planes con problemas: {fallos}"